# Exercise 2: Vocabulary overlap and NER

*Sequence Labelling · version 0.1.0*

> Fine-tune two models on KazNERD: one using `xlm-roberta-base` and one using `dbmdz/bert-base-turkish-cased` (BERTurk). Compare their micro and macro F1 (strict IOB2) on the KazNERD test set. Then measure BERTurk’s tokeniser coverage on a declared Cyrillic Kazakh word list. Report both the fraction of words containing an unknown token and mean subwords per word; a fragmented word is not an unknown word. Is the vocabulary gap a plausible explanation for the F1 difference, and what experiment would test it beyond this single pair of models?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "stanza==1.11.0" sklearn-crfsuite transformers sentencepiece requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`, which the first cell uses instead). Stanza is pinned to 1.11.0 because the worked solutions use its Kazakh KazNERD checkpoint; models already on disk are used from `STANZA_RESOURCES_DIR` (default `~/stanza_resources`), and missing ones are downloaded. The datasets are read from commit-pinned GitHub files (KazNERD, KyrgyzNER, the Starlang Turkish NER data, TRopBank, the Turkish WordNet and UD 2.17 IMST), downloaded once into `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`) and checked against SHA-256 hashes. Taggers receive the **gold words**, so tokenisation errors do not enter the scores.

**What is run here.** The tokeniser part runs for every tokeniser that can be loaded, from the Hugging Face Hub or from TurkicNLP's local model folder. Fine-tuning needs a GPU: `code/ch09-ner/train.py` is the chapter's complete recipe, and only `MODEL_NAME` changes between the two runs. The word list is every distinct word with at least one letter in the KazNERD test set.

In [ ]:
import collections, hashlib, os, random, re, statistics
from pathlib import Path
import requests

# Pinned public releases (commit-addressed raw files) and their SHA-256 hashes.
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_{}.txt"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/data/{}"
SHA256 = {
    KAZNERD.format("train"): "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KAZNERD.format("valid"): "29d584d1a0d9aca6d58b39eb8a93e154791d303ecff56ef93f12b3c889239bf1",
    KAZNERD.format("test"): "359a1f61f6a21280c6c47b4fde9a5adcd07411d62015aa6e32d940c51f163b7a",
    KYRGYZNER.format("KyrgyzNER_TRAIN.jsonl.txt"): "719a74a18cd99d6709c1e0592d0337aafade65aa97fe762e62f3f72a6b343a26",
    KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"): "57a35a92139497819a2792d3e9e00a41b387b029b7851be7a64f4968befaddf5",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()

def fetch(url, sha256=None):
    """Download a pinned file once, check its hash and return the local path."""
    sha256 = sha256 or SHA256[url]
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha256:
        raise ValueError(f"{url} differs from the pinned release")
    return path

def read_conll(path, tag_col=-1, comment="#"):
    """Sentences as (words, tags) from a column file with blank-line sentence breaks."""
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words:
                sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith(comment + " "):
            cols = line.split("\t") if "\t" in line else line.split()
            words.append(cols[0]); tags.append(cols[tag_col].strip())
    return sents

def spans(tags):
    """Strict IOB2 spans {(start, end, type)}; an I- tag that does not continue a span starts none."""
    out, start, typ = set(), None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if t.startswith("I-") and typ == t[2:]:
            continue
        if typ is not None:
            out.add((start, i, typ))
        start, typ = (i, t[2:]) if t.startswith("B-") else (None, None)
    return out

def bioes_to_iob2(tags):
    return ["B-" + t[2:] if t[:2] in ("S-", "B-") else "I-" + t[2:] if t[:2] in ("E-", "I-") else "O" for t in tags]

def prf(gold, pred):
    """Micro precision, recall and F1 (%) over lists of span sets."""
    g = sum(len(x) for x in gold); p = sum(len(x) for x in pred)
    c = sum(len(x & y) for x, y in zip(gold, pred))
    pr = c / p if p else 0.0; rc = c / g if g else 0.0
    return {"gold": g, "pred": p, "correct": c, "P": round(100 * pr, 2), "R": round(100 * rc, 2),
            "F1": round(100 * (2 * pr * rc / (pr + rc) if pr + rc else 0.0), 2)}

def stanza_ner(lang="kk"):
    """Stanza NER over gold words; models from STANZA_RESOURCES_DIR, downloaded only if missing."""
    import logging, stanza
    logging.getLogger("stanza").setLevel(logging.ERROR)
    model_dir = os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser()))
    have = (Path(model_dir) / lang / "ner").exists()
    return stanza.Pipeline(lang, processors="tokenize,ner", tokenize_pretokenized=True, model_dir=model_dir,
                           download_method=None if have else "reuse_resources", verbose=False)

def tag_words(nlp, sents, batch=2000):
    """IOB2 predictions for gold-word sentences (spaces inside a token are joined with '_')."""
    out = []
    for i in range(0, len(sents), batch):
        doc = nlp([[w.replace(" ", "_") for w in words] for words, _ in sents[i:i + batch]])
        for s, (words, _) in zip(doc.sentences, sents[i:i + batch]):
            assert len(s.tokens) == len(words)
            out.append(bioes_to_iob2([t.ner for t in s.tokens]))
    return out

In [ ]:
def load_tokenizer(name):
    """A Hugging Face tokenizer, from TurkicNLP's local model folder if present, otherwise from the Hub."""
    from transformers import AutoTokenizer
    local = Path("~/.turkicnlp/models/huggingface").expanduser() / name.replace("/", "--")
    return AutoTokenizer.from_pretrained(str(local) if (local / "tokenizer_config.json").exists() else name)

kaz_test = read_conll(fetch(KAZNERD.format("test")))
words = sorted({w for ws, _ in kaz_test for w in ws if any(ch.isalpha() for ch in w)})
print(f"{len(words)} distinct Kazakh word types (KazNERD test, words with at least one letter)")
print(f"{'tokeniser':16} {'words with UNK':>15} {'subwords/word':>14} {'single piece':>13}")
for label, name in [("BERTurk 32K", "dbmdz/bert-base-turkish-cased"), ("XLM-R base", "FacebookAI/xlm-roberta-base"),
                    ("Glot500 base", "cis-lmu/glot500-base"), ("NLLB-200", "facebook/nllb-200-distilled-600M")]:
    try:
        tok = load_tokenizer(name)
    except Exception as exc:
        print(f"{label:16} not available here ({type(exc).__name__})"); continue
    unk = tok.unk_token
    pieces = [tok.tokenize(w) for w in words]
    print(f"{label:16} {100 * sum(unk in p for p in pieces) / len(words):14.2f}% "
          f"{statistics.mean(len(p) for p in pieces):14.2f} {100 * sum(len(p) == 1 for p in pieces) / len(words):12.1f}%")

## Worked solution

Output of `experiments/ch09_exercises/run_ch09_exercises.py`:

```
21861 distinct Kazakh word types (KazNERD test, words with at least one letter)
tokeniser         words with UNK  subwords/word  single piece
BERTurk 32K      not available here (OSError)
XLM-R base       not available here (OSError)
Glot500 base               0.00%           2.44         11.3%
NLLB-200                   0.00%           2.64          8.6%
```

The two tokenisers of the exercise could not be loaded in the environment used for the worked solution, because the Hugging Face Hub was not reachable; the cell prints that instead of numbers, and the fine-tuning runs were not done. The two multilingual tokenisers available locally show the distinction the exercise asks for: neither produces an unknown token for any of the 21,861 Kazakh word types (0.00%), yet both fragment them, into 2.44 (Glot500) and 2.64 (NLLB-200) pieces per word, and only 11.3% and 8.6% of the words are a single piece.

**How to answer the question.** BERTurk's WordPiece vocabulary was built for Turkish Latin text, so Cyrillic letters that it does not contain become `[UNK]`, and the share of words with an unknown token is the first number to look at. A large unknown share explains an F1 gap better than fragmentation does, because an unknown token removes the word's identity. One pair of models cannot separate vocabulary from everything else that differs (pretraining data, model size, languages): a cleaner test keeps the model fixed and varies only the input, for example BERTurk on Kazakh Cyrillic against BERTurk on the same text transliterated to the Turkish alphabet, or XLM-R with and without a vocabulary extension.

## Your turn — extension

1. Where the Hub is reachable, rerun the cell to get the BERTurk and XLM-R rows, then train both models with `code/ch09-ner/train.py` and report micro and macro F1.
2. Transliterate the KazNERD test words to the Turkish alphabet and recompute BERTurk's unknown-token share.